# NuSMV baseline

This notebook runs the left- and right-turn examples from Yang et al., *Fine-Tuning Language Models Using Formal Methods Feedback*.

[Appendix D](https://arxiv.org/html/2310.18239v1#A4) · [Published supplement](https://proceedings.mlsys.org/paper_files/paper/2024/file/b0131b6ee02a00b03fc3320176fec8f5-Supplemental-Conference.pdf) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

Repaired code using apendix examples



### Setup
Open this notebook from the `paper_baseline` folder and run the cells in order using a Python 3 kernel. The portable Windows checker is included in `NuSMV`; keep that folder beside the notebook. No separate NuSMV installation is needed.


In [1]:
from pathlib import Path
import os
import re
import shutil
import subprocess
import tempfile

# Use the portable checker beside this notebook.
local_checker = Path.cwd() / "NuSMV" / "bin" / "NuSMV.exe"
NUSMV = str(local_checker.resolve()) if local_checker.is_file() else None
if not NUSMV:
    NUSMV = os.environ.get("NUSMV_PATH") or shutil.which("NuSMV")
if not NUSMV:
    raise FileNotFoundError(
        "Open the notebook from the paper_baseline folder. "
        "It should contain NuSMV/bin/NuSMV.exe."
    )
print("Using:", NUSMV)

Using: C:\Users\Kayla\Documents\Playground\paper_baseline\NuSMV\bin\NuSMV.exe


### Controllers
Each controller is checked in its own model. Its module is renamed `main` when the model is built. 

In [2]:
left_before = """
MODULE turn_left_before_finetune

VAR
    green_traffic_light : boolean;
    green_left_turn_light : boolean;
    opposite_car : boolean;
    car_from_left : boolean;
    car_from_right : boolean;
    pedestrian_at_left : boolean;
    pedestrian_at_right : boolean;
    side_car : boolean;
    stop_sign : boolean;
    action : {stop, turn_left, turn_right, go_straight};

ASSIGN
    init(action) := stop;

TRANS
    case
        !green_left_turn_light : next(action) = stop;
        green_left_turn_light & !opposite_car & !car_from_left & !car_from_right & !pedestrian_at_left & !pedestrian_at_right : next(action) = turn_left;
        opposite_car | car_from_left | car_from_right | pedestrian_at_left | pedestrian_at_right : next(action) = stop;
        action = turn_left : next(action) = go_straight;
        TRUE : next(action) = stop;
    esac;
"""

left_after = """
MODULE turn_left_after_finetune

VAR
    green_traffic_light : boolean;
    green_left_turn_light : boolean;
    opposite_car : boolean;
    car_from_left : boolean;
    car_from_right : boolean;
    pedestrian_at_left : boolean;
    pedestrian_at_right : boolean;
    side_car : boolean;
    stop_sign : boolean;
    action : {stop, turn_left, turn_right, go_straight};

ASSIGN
    init(action) := stop;

TRANS
    case
        !green_left_turn_light : next(action) = stop;
        green_left_turn_light : next(action) = turn_left;
    esac;
"""

In [3]:
right_before = """
MODULE turn_right_before_finetune

VAR
    green_traffic_light : boolean;
    green_left_turn_light : boolean;
    opposite_car : boolean;
    car_from_left : boolean;
    car_from_right : boolean;
    pedestrian_at_left : boolean;
    pedestrian_at_right : boolean;
    side_car : boolean;
    stop_sign : boolean;
    action : {stop, turn_left, turn_right, go_straight};

ASSIGN
    init(action) := stop;
    next(action) :=
        case
            green_traffic_light & !car_from_right : {go_straight, turn_right};
            green_traffic_light & car_from_right : go_straight;
            !green_traffic_light : stop;
        esac;
"""

right_after = """
MODULE turn_right_after_finetune
VAR
    green_traffic_light : boolean;
    green_left_turn_light : boolean;
    opposite_car : boolean;
    car_from_left : boolean;
    car_from_right : boolean;
    pedestrian_at_left : boolean;
    pedestrian_at_right : boolean;
    side_car : boolean;
    stop_sign : boolean;
    action : {stop, turn_left, turn_right, go_straight};
ASSIGN
    init(action) := stop;
TRANS
    case
        !car_from_left & !pedestrian_at_right : next(action) = turn_right;
        car_from_left | pedestrian_at_right : next(action) = stop;
    esac;
"""

### Properties
Both versions of a controller get the same checks.

For the left-turn sample property the undefined `left_turn_light in {flashing, green}` is replaced with the declared Boolean `green_left_turn_light`. This assumes that the Boolean represents permission to turn. It doesnt model a separate flashing signal while The right-turn negation is parenthesized for clarity.

The `sample_` checks preserve the paper's timing. The extra `next_` checks are added here: `X` checks the next action against the current observation. They help distinguish a controller decision from an input that changes on the next step.

Inputs remain unconstrained and No environment automaton or fairness assumptions are invented. ex: the light can remain red forever so recurring left turns arent guaranteed.

In [4]:
properties = {
    "left": {
        "sample_phi_1": "G (action = turn_left -> green_left_turn_light)",
        "sample_phi_2": "G (F (action = turn_left))",
        "next_red_stop": "G (!green_left_turn_light -> X (action = stop))",
        "next_green_turn": "G (green_left_turn_light -> X (action = turn_left))",
    },
    "right": {
        "sample_phi_1": "G (pedestrian_at_right -> !(action = turn_right))",
        "sample_phi_2": "G (car_from_left -> !(action = turn_right))",
        "next_pedestrian_safe": "G (pedestrian_at_right -> X (!(action = turn_right)))",
        "next_car_safe": "G (car_from_left -> X (!(action = turn_right)))",
    },
}
controllers = {
    "left_before": left_before,
    "left_after": left_after,
    "right_before": right_before,
    "right_after": right_after,
}

def make_model(name):
    direction = name.split("_")[0]
    text = re.sub(r"MODULE \w+", "MODULE main", controllers[name], count=1)
    for label, formula in properties[direction].items():
        text += f"\nLTLSPEC NAME {label} := {formula};\n"
    return text

print(make_model("right_before"))


MODULE main

VAR
    green_traffic_light : boolean;
    green_left_turn_light : boolean;
    opposite_car : boolean;
    car_from_left : boolean;
    car_from_right : boolean;
    pedestrian_at_left : boolean;
    pedestrian_at_right : boolean;
    side_car : boolean;
    stop_sign : boolean;
    action : {stop, turn_left, turn_right, go_straight};

ASSIGN
    init(action) := stop;
    next(action) :=
        case
            green_traffic_light & !car_from_right : {go_straight, turn_right};
            green_traffic_light & car_from_right : go_straight;
            !green_traffic_light : stop;
        esac;

LTLSPEC NAME sample_phi_1 := G (pedestrian_at_right -> !(action = turn_right));

LTLSPEC NAME sample_phi_2 := G (car_from_left -> !(action = turn_right));

LTLSPEC NAME next_pedestrian_safe := G (pedestrian_at_right -> X (!(action = turn_right)));

LTLSPEC NAME next_car_safe := G (car_from_left -> X (!(action = turn_right)));



### Run the checks
This uses NuSMV's LTL model checker so each property is selected by its actual name. if a checker error stops the run it is not counted as a failed safety property.

In [5]:
results = []
logs = {}

def check_model(name):
    direction = name.split("_")[0]
    labels = list(properties[direction])
    with tempfile.TemporaryDirectory(prefix="paper_nusmv_") as folder:
        folder = Path(folder)
        (folder / "model.smv").write_text(make_model(name), encoding="utf-8")
        commands = ["read_model -i model.smv", "go", "check_fsm"]
        commands += [f'check_ltlspec -P "{label}"' for label in labels]
        commands += ["quit"]
        (folder / "check.cmd").write_text("\n".join(commands) + "\n", encoding="utf-8")
        run = subprocess.run(
            [str(NUSMV), "-source", "check.cmd"], cwd=folder,
            capture_output=True, text=True, timeout=120,
        )
    output = run.stdout + "\n" + run.stderr
    logs[name] = output
    verdicts = re.findall(r"^-- specification .* is (true|false)\s*$", run.stdout, re.MULTILINE)
    if run.returncode != 0 or len(verdicts) != len(labels):
        raise RuntimeError(f"NuSMV did not complete {name}:\n{output}")
    if "transition relation is total" not in output.lower():
        raise RuntimeError(f"Check the model for deadlocks:\n{output}")
    return [{"controller": name, "property": label,
             "source": "sample (repaired)" if label.startswith("sample_") else "added timing check",
             "passed": value == "true"}
            for label, value in zip(labels, verdicts)]

for name in controllers:
    results.extend(check_model(name))

print(f"{'Controller':<15} {'Property':<23} Result")
print("-" * 48)
for row in results:
    verdict = "PASS" if row["passed"] else "FAIL"
    print(f"{row['controller']:<15} {row['property']:<23} {verdict}")

Controller      Property                Result
------------------------------------------------
left_before     sample_phi_1            FAIL
left_before     sample_phi_2            FAIL
left_before     next_red_stop           PASS
left_before     next_green_turn         FAIL
left_after      sample_phi_1            FAIL
left_after      sample_phi_2            FAIL
left_after      next_red_stop           PASS
left_after      next_green_turn         PASS
right_before    sample_phi_1            FAIL
right_before    sample_phi_2            FAIL
right_before    next_pedestrian_safe    FAIL
right_before    next_car_safe           FAIL
right_after     sample_phi_1            FAIL
right_after     sample_phi_2            FAIL
right_after     next_pedestrian_safe    PASS
right_after     next_car_safe           PASS


### Read a counterexample
`FAIL` means NuSMV found an allowed behavior that violates the property. It does not mean the checker crashed. The full output below includes the counterexample states. Change the key to inspect another controller.

In [6]:
print(logs["right_after"])

*** This is NuSMV 2.7.1 (compiled on Mon Sep 29 16:07:59 2025)
*** Enabled addons are: compass
*** For more information on NuSMV see <http://nusmv.fbk.eu>
*** or email to <nusmv-users@list.fbk.eu>.
*** Please report bugs to <Please report bugs to <nusmv-users@fbk.eu>>

*** Copyright (c) 2010-2025, Fondazione Bruno Kessler

*** This version of NuSMV is linked to the CUDD library version 2.4.1
*** Copyright (c) 1995-2004, Regents of the University of Colorado

*** This version of NuSMV is linked to the MiniSat SAT solver. 
*** See http://minisat.se/MiniSat.html
*** Copyright (c) 2003-2006, Niklas Een, Niklas Sorensson
*** Copyright (c) 2007-2010, Niklas Sorensson


##########################################################
The transition relation is total: No deadlock state exists
##########################################################
-- specification  G (pedestrian_at_right -> !(action = turn_right))  is false
-- as demonstrated by the following execution sequence
Trace Description:

### Notes
- The current state safety checks can fail because an input changes after the action was selected. The added next action checks make timing explicit.
- A left-turn liveness failure is possible with an indefinitely red light. No assumption forcing a green light has been added.
- An “after” controller is not guaranteed to pass every property.

The unchanged code is located in `paper_code_only.ipynb`.